# 03 — Speaker Diarization | Maharah Call Center Agent

**Inputs:** `Maharah_Processed_Data.zip` from the ASR notebook, and a private Colab Secret named `HF_TOKEN`.

**Outputs:** diarization segments, DER vs dataset references, word-aligned transcripts, and results for 12 calls. **Keep original WAV files out of GitHub.**

Before starting: Colab **Runtime → Change runtime type → T4 GPU**. Run cells in order. Download output ZIP before ending the Colab session.

**Integrity note:** Pyannote predicts anonymous speaker IDs only. `agent`/`customer` below are assigned with *dataset reference-assisted mapping* **after** inference, not identified independently by Pyannote. DER uses unmodified anonymous speaker predictions.

## Cell 1 — Install Dependencies

Install the required libraries for speaker diarization, audio processing, and evaluation using Pyannote Community-1.

In [2]:
%pip install -q "pyannote.audio>=4.0,<5.0" huggingface_hub soundfile pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## Cell 2 — Imports and Environment Setup

Import the required libraries, configure project directories, and verify GPU availability.

This step ensures that the environment is ready for speaker diarization using Pyannote Community-1.

In [2]:
# ============================================================
# Cell 2 — Imports and Environment Setup
# ============================================================

# Standard libraries
import json
import zipfile
import time
import re

from pathlib import Path
from itertools import permutations

# Data processing
import numpy as np
import pandas as pd
import soundfile as sf

# Deep learning
import torch

# Google Colab
from google.colab import files, userdata

# Hugging Face
from huggingface_hub import hf_hub_download

# Speaker diarization
import pyannote.audio
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate

# Display
from IPython.display import display


# ============================================================
# Project Directories
# ============================================================

ROOT = Path("/content")

# Previous ASR results
ASR_DIR = ROOT / "data/processed/asr/small_en_cpu"

# Selected calls metadata
MANIFEST = ROOT / "data/processed/selected_calls.csv"

# Speaker diarization output
OUT_DIR = ROOT / "data/processed/diarization/community_1"

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Device Configuration
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# Environment Verification
# ============================================================

print("=" * 60)
print("SPEAKER DIARIZATION — ENVIRONMENT CHECK")
print("=" * 60)

print(f"PyTorch Version:  {torch.__version__}")
print(f"NumPy Version:    {np.__version__}")
print(f"Pyannote Version: {pyannote.audio.__version__}")

print("-" * 60)

print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
else:
    print("GPU:              Not Available")

print("-" * 60)

print(f"Project Root:     {ROOT}")
print(f"ASR Directory:    {ASR_DIR}")
print(f"Output Directory: {OUT_DIR}")

print("=" * 60)
print("Environment Setup Completed Successfully!")
print("=" * 60)

SPEAKER DIARIZATION — ENVIRONMENT CHECK
PyTorch Version:  2.11.0+cu130
NumPy Version:    2.5.4
Pyannote Version: 4.0.7
------------------------------------------------------------
CUDA Available:   True
GPU:              Tesla T4
------------------------------------------------------------
Project Root:     /content
ASR Directory:    /content/data/processed/asr/small_en_cpu
Output Directory: /content/data/processed/diarization/community_1
Environment Setup Completed Successfully!


## Cell 3 — Upload and restore saved ASR results

In [3]:
if not MANIFEST.exists() or not (ASR_DIR / "CALL_001.json").exists():
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    assert zip_names, "Upload Maharah_Processed_Data.zip"
    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall(ROOT)

assert MANIFEST.exists(), "Missing selected_calls.csv"
assert (ASR_DIR / "run_config.json").exists(), "Missing ASR run_config.json"

selected_calls = pd.read_csv(MANIFEST)
with open(ASR_DIR / "run_config.json", encoding="utf-8") as f:
    asr_config = json.load(f)

DATASET_REPO = asr_config["dataset"]
DATASET_REV = asr_config["dataset_revision"]

print("Selected calls:", len(selected_calls))
print("ASR JSON files:", len(list(ASR_DIR.glob("CALL_???.json"))))
print("Dataset:", DATASET_REPO)
print("Pinned revision:", DATASET_REV)
display(selected_calls[["call_id", "accent", "domain", "duration_sec"]])

Saving Maharah_Processed_Data.zip to Maharah_Processed_Data (1).zip
Selected calls: 12
ASR JSON files: 12
Dataset: apptek-com/apptek_callcenter_dialogues
Pinned revision: b98967d9946f7f59f58d08624a2a00fe98fe0219


,call_id,accent,domain,duration_sec
0,CALL_001,en-US_General,food,386.637
1,CALL_002,en-US_General,hospitality,630.749
2,CALL_003,en-US_General,entertainment,701.906
3,CALL_004,en-GB,hospitality,421.525
4,CALL_005,en-GB,retail,611.108
5,CALL_006,en-GB,entertainment,603.482
6,CALL_007,en-IN,telecom,631.712
7,CALL_008,en-IN,insurance,625.664
8,CALL_009,en-IN,health,422.208
9,CALL_010,en-AU,banking,420.000


## Cell 4 — Authenticate securely and load Pyannote Community-1

In [4]:
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN to Colab Secrets and enable Notebook access."

MODEL_ID = "pyannote/speaker-diarization-community-1"
pipeline = Pipeline.from_pretrained(MODEL_ID, token=hf_token)
assert pipeline is not None, "Model not loaded: check Hugging Face access and token."
pipeline.to(DEVICE)

print("Model loaded:", MODEL_ID)
print("Device:", DEVICE)
# Do not print or save your token.

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

Model loaded: pyannote/speaker-diarization-community-1
Device: cuda


## Cell 5 — Reusable functions (audio, diarization, DER, transcript alignment)

In [5]:
def download_audio(row):
    relative_path = f"diarization/{row['accent']}/{row['file_name']}"
    return Path(hf_hub_download(
        repo_id=DATASET_REPO, repo_type="dataset",
        revision=DATASET_REV, filename=relative_path,
        local_dir=str(ROOT / "data/raw/apptek"),
    ))


def load_reference(row):
    """Download evaluation labels AFTER diarization prediction."""
    metadata_path = str(row["metadata_path"])
    local_path = hf_hub_download(
        repo_id=DATASET_REPO, repo_type="dataset",
        revision=DATASET_REV, filename=metadata_path,
        local_dir=str(ROOT / "data/raw/apptek"),
    )
    with open(local_path, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            if item.get("file_name") == str(row["file_name"]):
                return item
    raise ValueError(f"Reference not found: {row['file_name']}")


def to_segments(annotation):
    return [
        {"start": round(float(seg.start), 3),
         "end": round(float(seg.end), 3), "speaker": str(speaker)}
        for seg, _, speaker in annotation.itertracks(yield_label=True)
    ]


def make_reference_annotation(reference_item):
    annotation = Annotation()
    for i, item in enumerate(reference_item["segments"]):
        start, end = float(item["start"]), float(item["end"])
        if end > start:
            annotation[Segment(start, end), i] = item["role"]
    return annotation


def role_mapping_from_reference(predicted_segments, gold_segments):
    """Reference-assisted roles (NOT a model-predicted agent/customer classifier)."""
    speakers = sorted(set(x["speaker"] for x in predicted_segments))
    roles = sorted(set(x["role"] for x in gold_segments))
    if len(speakers) != 2 or len(roles) != 2:
        return {speaker: "Unknown" for speaker in speakers}

    overlap = {(speaker, role): 0.0 for speaker in speakers for role in roles}
    for pred in predicted_segments:
        for gold in gold_segments:
            shared = max(0.0, min(pred["end"], gold["end"]) - max(pred["start"], gold["start"]))
            overlap[pred["speaker"], gold["role"]] += shared

    best = max(
        permutations(roles),
        key=lambda candidate: sum(overlap[spk, role] for spk, role in zip(speakers, candidate)),
    )
    return dict(zip(speakers, best))


def assign_speaker_to_word(word, exclusive_segments):
    start, end = float(word["start"]), float(word["end"])
    overlaps = [
        (max(0.0, min(end, x["end"]) - max(start, x["start"])), x["speaker"])
        for x in exclusive_segments
    ]
    if not overlaps:
        return "Unknown"
    best_overlap, speaker = max(overlaps)
    if best_overlap > 0:
        return speaker
    midpoint = (start + end) / 2
    nearest = min(
        exclusive_segments,
        key=lambda x: max(x["start"] - midpoint, midpoint - x["end"], 0.0),
    )
    return nearest["speaker"]


def align_asr_words(asr_json, exclusive_segments, role_map):
    words = [word for segment in asr_json["predicted_segments"]
             for word in segment.get("words", [])
             if word.get("start") is not None and word.get("end") is not None]
    words.sort(key=lambda x: (x["start"], x["end"]))

    turns = []
    for word in words:
        speaker = assign_speaker_to_word(word, exclusive_segments)
        start, end = float(word["start"]), float(word["end"])
        text = str(word["word"]).strip()
        if not text:
            continue
        if turns and turns[-1]["speaker"] == speaker and start - turns[-1]["end"] <= 1.0:
            turns[-1]["text"] += " " + text
            turns[-1]["end"] = max(turns[-1]["end"], end)
        else:
            turns.append({"start": start, "end": end,
                          "speaker": speaker, "role": role_map.get(speaker, "Unknown"),
                          "text": text})

    for turn in turns:
        turn["start"] = round(turn["start"], 2)
        turn["end"] = round(turn["end"], 2)
        turn["text"] = re.sub(r"\s+([,.!?;:])", r"\1", turn["text"])
    return turns


def run_call(row, force=False):
    call_id = str(row["call_id"])
    result_path = OUT_DIR / f"{call_id}.json"
    if result_path.exists() and not force:
        with open(result_path, encoding="utf-8") as f:
            return json.load(f)

    audio_path = download_audio(row)
    audio, sample_rate = sf.read(str(audio_path), dtype="float32", always_2d=True)
    waveform = torch.from_numpy(audio.T.copy())  # (channels, samples)
    # Loading an in-memory waveform avoids TorchCodec/FFmpeg decoding issues.
    print(f"Running {call_id} | sample rate={sample_rate}, channels={waveform.shape[0]}")

    t0 = time.perf_counter()
    with torch.inference_mode():
        result = pipeline({"waveform": waveform, "sample_rate": int(sample_rate)}, num_speakers=2)
    elapsed = time.perf_counter() - t0
    del waveform, audio

    # Predictions are frozen BEFORE fetching the reference annotations.
    regular = to_segments(result.speaker_diarization)
    exclusive = to_segments(result.exclusive_speaker_diarization)
    reference = load_reference(row)
    ref_annotation = make_reference_annotation(reference)
    der_metric = DiarizationErrorRate(collar=0.0, skip_overlap=False)
    der = float(der_metric(ref_annotation, result.speaker_diarization))
    role_map = role_mapping_from_reference(regular, reference["segments"])

    with open(ASR_DIR / f"{call_id}.json", encoding="utf-8") as f:
        asr = json.load(f)
    turns = align_asr_words(asr, exclusive, role_map)
    output = {
        "call_id": call_id, "accent": row["accent"], "domain": row["domain"],
        "model": MODEL_ID, "dataset_revision": DATASET_REV,
        "duration_sec": float(row["duration_sec"]),
        "inference_sec": round(elapsed, 2),
        "der": round(der, 6),
        "der_settings": {"collar": 0.0, "skip_overlap": False,
                         "evaluated_with": "dataset reference role annotations"},
        "role_label_source": "reference-assisted mapping, NOT model inference",
        "speaker_to_role": role_map,
        "predicted_diarization": regular,
        "exclusive_diarization": exclusive,
        "asr_aligned_turns": turns,
    }
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    pd.DataFrame(turns, columns=["start", "end", "speaker", "role", "text"]).to_csv(
        OUT_DIR / f"{call_id}_turns.csv", index=False)
    print(f"Saved {call_id} | DER={der:.2%} | turns={len(turns)} | {elapsed:.1f}s")
    return output

## Cell 6 — Run pilot on CALL_001 and evaluate DER

In [6]:
pilot_row = selected_calls.loc[selected_calls["call_id"] == "CALL_001"].iloc[0]
pilot = run_call(pilot_row)
print("\nCALL_001 DER:", f"{pilot['der']:.2%}")
print("Anonymous speaker → reference-assisted role:", pilot["speaker_to_role"])
print("\nFirst diarization segments:")
display(pd.DataFrame(pilot["predicted_diarization"]).head(12))

diarization/en-US_General/audio/en_US_Ge(…): reconstructing file:   0%|          |  0.00B / 12.4MB            

diarization/en-US_General/audio/en_US_Ge(…): downloading bytes:           |  0.00B            

Running CALL_001 | sample rate=16000, channels=1


/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


metadata.jsonl:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

Saved CALL_001 | DER=6.94% | turns=31 | 20.6s

CALL_001 DER: 6.94%
Anonymous speaker → reference-assisted role: {'SPEAKER_00': 'customer', 'SPEAKER_01': 'agent'}

First diarization segments:


/usr/local/lib/python3.13/dist-packages/pyannote/metrics/utils.py:199: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


,start,end,speaker
0,3.322,3.710,SPEAKER_00
1,11.894,19.842,SPEAKER_01
2,20.703,26.440,SPEAKER_00
3,27.166,37.392,SPEAKER_01
4,40.058,51.432,SPEAKER_00
5,52.664,54.031,SPEAKER_01
6,54.284,55.651,SPEAKER_01
7,55.955,56.731,SPEAKER_01
8,57.305,58.300,SPEAKER_01
9,58.672,60.578,SPEAKER_01


## Cell 7 — Inspect aligned transcript

In [7]:
print("First 20 aligned transcript turns:")
display(pd.DataFrame(pilot["asr_aligned_turns"]).head(20))
print("\nIMPORTANT: 'agent/customer' labels are reference-assisted; Pyannote predicted only speaker IDs.")

First 20 aligned transcript turns:


,start,end,speaker,role,text
0,11.44,19.74,SPEAKER_01,agent,ring ring hi there this is Leo from feast and ...
1,19.74,26.28,SPEAKER_00,customer,hi Leo yeah this is Ashley I'm looking into co...
2,26.28,37.28,SPEAKER_01,agent,oh I love a good team lunch nothing brings co ...
3,39.29,51.21,SPEAKER_00,customer,Wow you sound very enthusiastic about this I l...
4,51.21,65.57,SPEAKER_01,agent,challenge accepted vegan check gluten free dou...
5,65.57,72.93,SPEAKER_00,customer,perfect um I also wanted to know about pricing...
6,74.49,84.59,SPEAKER_01,agent,"Numbers and norms, my two favorite things. Our..."
7,85.71,97.95,SPEAKER_01,agent,We've got a little bit of everything. Think qu...
8,99.35,101.45,SPEAKER_00,customer,"Wow, all of that sounds amazing."
9,102.97,123.91,SPEAKER_00,customer,"Yeah, I think that the canua salad would be a ..."



IMPORTANT: 'agent/customer' labels are reference-assisted; Pyannote predicted only speaker IDs.


## Cell 8 — Run all 12 calls (optional, after reviewing pilot)

Change `RUN_ALL = False` to `True`. Calls already saved (including CALL_001) are skipped, so the notebook can resume after interruptions.

In [8]:
RUN_ALL = True  # Change to True only after reviewing CALL_001.

if RUN_ALL:
    errors = {}
    for _, row in selected_calls.iterrows():
        try:
            run_call(row)
        except Exception as e:
            errors[str(row["call_id"])] = repr(e)
            print("ERROR:", row["call_id"], e)

    print("Finished. Errors:", errors)

results = []
for path in sorted(OUT_DIR.glob("CALL_???.json")):
    with open(path, encoding="utf-8") as f:
        item = json.load(f)
    results.append({"call_id": item["call_id"], "accent": item["accent"],
                    "domain": item["domain"], "der": item["der"],
                    "inference_sec": item["inference_sec"],
                    "aligned_turns": len(item["asr_aligned_turns"])})

summary = pd.DataFrame(results)
if not summary.empty:
    summary.to_csv(OUT_DIR / "diarization_metrics.csv", index=False)
    display(summary)
    print("Macro average DER:", f"{summary['der'].mean():.2%}")
print(f"Saved: {len(results)}/{len(selected_calls)} calls")

diarization/en-US_General/audio/en_US_Ge(…): reconstructing file:   0%|          |  0.00B / 20.2MB            

diarization/en-US_General/audio/en_US_Ge(…): downloading bytes:           |  0.00B            

Running CALL_002 | sample rate=16000, channels=1


/usr/local/lib/python3.13/dist-packages/pyannote/metrics/utils.py:199: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


Saved CALL_002 | DER=9.78% | turns=48 | 26.3s


diarization/en-US_General/audio/en_US_Ge(…): reconstructing file:   0%|          |  0.00B / 22.5MB            

... (49 outputs left out)

Running CALL_012 | sample rate=16000, channels=1
Saved CALL_012 | DER=7.28% | turns=55 | 37.5s
Finished. Errors: {}


/usr/local/lib/python3.13/dist-packages/pyannote/metrics/utils.py:199: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


,call_id,accent,domain,der,inference_sec,aligned_turns
0,CALL_001,en-US_General,food,0.069391,20.61,31
1,CALL_002,en-US_General,hospitality,0.097804,26.33,48
2,CALL_003,en-US_General,entertainment,0.211123,29.75,97
3,CALL_004,en-GB,hospitality,0.147297,18.10,51
4,CALL_005,en-GB,retail,0.550250,25.89,78
5,CALL_006,en-GB,entertainment,0.508983,25.79,66
6,CALL_007,en-IN,telecom,0.228521,26.86,105
7,CALL_008,en-IN,insurance,0.168550,26.69,77
8,CALL_009,en-IN,health,0.171854,18.22,66
9,CALL_010,en-AU,banking,0.098316,17.95,52


Macro average DER: 19.93%
Saved: 12/12 calls


## Cell 9 — Download results ZIP

In [16]:
zip_path = ROOT / "Maharah_Diarization_Results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(ROOT))

print("Results ZIP:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
files.download(str(zip_path))

Results ZIP: /content/Maharah_Diarization_Results.zip
Size (MB): 0.12


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 10 — Diarization Error Analysis

Investigate the two calls with the highest Diarization Error Rate (DER): CALL_005 and CALL_006.

Break down the diarization errors into:
- Missed Detection
- False Alarm
- Speaker Confusion

Identify the main sources of diarization errors without rerunning the model.

In [11]:
# ============================================================
# Cell 10 — Diarization Error Analysis
# ============================================================

TARGET_CALLS = ["CALL_005", "CALL_006"]


def segments_to_annotation(segments, role_map=None):

    annotation = Annotation()

    for i, item in enumerate(segments):

        start = float(item["start"])
        end = float(item["end"])

        if end <= start:
            continue

        speaker = str(item["speaker"])

        if role_map is not None:
            speaker = role_map.get(speaker, speaker)

        annotation[Segment(start, end), i] = speaker

    return annotation


def load_saved_analysis(call_id):

    # Load saved diarization output
    path = OUT_DIR / f"{call_id}.json"

    with open(path, encoding="utf-8") as f:
        result = json.load(f)

    # Find call metadata
    row = selected_calls.loc[
        selected_calls["call_id"] == call_id
    ].iloc[0]

    # Load reference labels
    reference_data = load_reference(row)

    reference = make_reference_annotation(
        reference_data
    )

    # Reconstruct predicted diarization
    prediction = segments_to_annotation(
        result["predicted_diarization"]
    )

    return row, result, reference, prediction


analysis_results = []

for call_id in TARGET_CALLS:

    row, result, reference, prediction = (
        load_saved_analysis(call_id)
    )

    # Detailed DER evaluation
    metric = DiarizationErrorRate(
        collar=0.0,
        skip_overlap=False
    )

    details = metric(
        reference,
        prediction,
        detailed=True
    )

    total = float(details["total"])

    missed = float(details["missed detection"])
    false_alarm = float(details["false alarm"])
    confusion = float(details["confusion"])

    der = (
        (missed + false_alarm + confusion) / total
        if total > 0 else float("nan")
    )

    components = {
        "Missed Detection": missed,
        "False Alarm": false_alarm,
        "Speaker Confusion": confusion
    }

    main_error = max(
        components,
        key=components.get
    )

    analysis_results.append({
        "call_id": call_id,
        "saved_der_pct": result["der"] * 100,
        "recalculated_der_pct": der * 100,
        "missed_detection_pct": missed / total * 100,
        "false_alarm_pct": false_alarm / total * 100,
        "speaker_confusion_pct": confusion / total * 100,
        "main_error": main_error
    })


error_analysis_df = pd.DataFrame(
    analysis_results
)

display(
    error_analysis_df.round(2)
)

# Save analysis
error_analysis_df.to_csv(
    OUT_DIR / "failure_analysis_breakdown.csv",
    index=False
)

print("=" * 60)
print("DIARIZATION ERROR ANALYSIS COMPLETED")
print("=" * 60)

/usr/local/lib/python3.13/dist-packages/pyannote/metrics/utils.py:199: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


,call_id,saved_der_pct,recalculated_der_pct,missed_detection_pct,false_alarm_pct,speaker_confusion_pct,main_error
0,CALL_005,55.02,55.03,23.14,0.53,31.35,Speaker Confusion
1,CALL_006,50.90,50.90,16.94,0.53,33.43,Speaker Confusion


DIARIZATION ERROR ANALYSIS COMPLETED


## Cell 11 — Identify High-Error Time Windows

Analyze CALL_005 and CALL_006 using 30-second intervals.

Identify the time windows with the highest speaker attribution and speech detection errors.

Use fixed, reference-assisted speaker mappings for diagnostic evaluation without reassigning speaker identities independently in each window.

In [12]:
# ============================================================
# Cell 11 — Identify High-Error Time Windows
# ============================================================

from pyannote.core import Timeline
from pyannote.metrics.identification import IdentificationErrorRate

WINDOW_SIZE = 30.0

window_results = []

for call_id in TARGET_CALLS:

    row, result, reference, prediction = (
        load_saved_analysis(call_id)
    )

    # Apply the existing reference-assisted role mapping
    mapped_prediction = segments_to_annotation(
        result["predicted_diarization"],
        role_map=result["speaker_to_role"]
    )

    duration = float(result["duration_sec"])

    for start in np.arange(0, duration, WINDOW_SIZE):

        end = min(start + WINDOW_SIZE, duration)

        evaluation_region = Timeline([
            Segment(float(start), float(end))
        ])

        # Fixed speaker labels across windows
        metric = IdentificationErrorRate(
            collar=0.0,
            skip_overlap=False
        )

        details = metric(
            reference,
            mapped_prediction,
            uem=evaluation_region,
            detailed=True
        )

        total = float(details["total"])

        if total <= 0:
            continue

        missed = float(details["missed detection"])
        false_alarm = float(details["false alarm"])
        confusion = float(details["confusion"])

        error_rate = (
            (missed + false_alarm + confusion)
            / total * 100
        )

        window_results.append({
            "call_id": call_id,
            "start": round(float(start), 2),
            "end": round(float(end), 2),
            "window_error_pct": round(error_rate, 2),
            "missed_sec": round(missed, 2),
            "false_alarm_sec": round(false_alarm, 2),
            "confusion_sec": round(confusion, 2)
        })


window_df = pd.DataFrame(window_results)

# Identify the three worst windows per call
worst_windows = (
    window_df
    .sort_values(
        ["call_id", "window_error_pct"],
        ascending=[True, False]
    )
    .groupby("call_id")
    .head(3)
    .reset_index(drop=True)
)

print("=" * 60)
print("TOP 3 HIGHEST-ERROR WINDOWS PER CALL")
print("=" * 60)

display(worst_windows)

# Save results
window_df.to_csv(
    OUT_DIR / "window_error_analysis.csv",
    index=False
)

worst_windows.to_csv(
    OUT_DIR / "worst_error_windows.csv",
    index=False
)

TOP 3 HIGHEST-ERROR WINDOWS PER CALL


,call_id,start,end,window_error_pct,missed_sec,false_alarm_sec,confusion_sec
0,CALL_005,300.0,330.0,86.22,7.91,0.04,17.02
1,CALL_005,30.0,60.0,77.40,5.65,0.00,13.45
2,CALL_005,0.0,30.0,76.26,1.38,0.02,10.79
3,CALL_006,570.0,600.0,100.65,2.94,0.08,8.90
4,CALL_006,300.0,330.0,100.17,1.39,0.02,10.61
5,CALL_006,30.0,60.0,80.29,3.43,0.11,20.34


## Cell 12 — Inspect Problematic Conversation Segments

Inspect the most problematic 30-second interval from each call.

Review:
- Speaker labels
- Agent and customer roles
- Whisper transcripts
- Original audio recordings

Use manual inspection to investigate possible causes of high diarization errors.

In [13]:
# ============================================================
# Cell 12 — Inspect Problematic Conversation Segments
# ============================================================

from IPython.display import Audio

for call_id in TARGET_CALLS:

    print("\n" + "=" * 65)
    print(f"MANUAL REVIEW — {call_id}")
    print("=" * 65)

    # Select the highest-error window
    call_windows = worst_windows[
        worst_windows["call_id"] == call_id
    ]

    if call_windows.empty:
        print("No valid evaluation windows.")
        continue

    worst = call_windows.iloc[0]

    start = float(worst["start"])
    end = float(worst["end"])

    print(f"Time: {start:.1f}s - {end:.1f}s")

    print(
        f"Window Error: "
        f"{worst['window_error_pct']:.2f}%"
    )

    # Load saved transcript
    result_path = OUT_DIR / f"{call_id}.json"

    with open(
        result_path,
        encoding="utf-8"
    ) as f:

        result = json.load(f)

    turns = pd.DataFrame(
        result["asr_aligned_turns"]
    )

    # Find overlapping transcript turns
    selected_turns = turns[
        (turns["end"] > start) &
        (turns["start"] < end)
    ]

    print("\nTranscript:")

    display(
        selected_turns[
            ["start", "end", "speaker", "role", "text"]
        ]
    )

    # Load original audio
    row = selected_calls.loc[
        selected_calls["call_id"] == call_id
    ].iloc[0]

    audio_path = download_audio(row)

    audio_info = sf.info(str(audio_path))

    sample_rate = audio_info.samplerate

    audio_clip, _ = sf.read(
        str(audio_path),
        start=int(start * sample_rate),
        stop=int(end * sample_rate),
        dtype="float32"
    )

    print("\nOriginal Audio:")

    display(
        Audio(
            audio_clip.T,
            rate=sample_rate
        )
    )


MANUAL REVIEW — CALL_005
Time: 300.0s - 330.0s
Window Error: 86.22%

Transcript:


,start,end,speaker,role,text
42,299.53,300.99,SPEAKER_00,customer,and
43,303.32,306.52,SPEAKER_00,customer,just some designer
44,307.84,311.38,SPEAKER_00,customer,shoes and other ladies wears.
45,312.44,317.66,SPEAKER_01,agent,I think I'm gonna have to come and still to ha...
46,320.44,333.70,SPEAKER_01,agent,"So could you also, I'm also hoping to purchase..."



Original Audio:



MANUAL REVIEW — CALL_006
Time: 570.0s - 600.0s
Window Error: 100.65%

Transcript:


,start,end,speaker,role,text
61,569.13,570.65,SPEAKER_01,customer,Sure. My name is Melissa.
62,574.49,574.89,SPEAKER_01,customer,Sure.
63,576.23,582.13,SPEAKER_01,customer,I'll send you your email confirmation and text...
64,584.51,586.07,SPEAKER_01,customer,You too. Have a day. Thank you.
65,587.61,588.07,SPEAKER_01,customer,Bye.



Original Audio:


## Cell 13 — Reference vs Predicted Speaker Comparison

Compare the reference speaker annotations with Pyannote predictions
in the highest-error windows of CALL_005 and CALL_006.

The analysis helps identify speaker confusion, missed speech,
and possible speaker attribution inconsistencies.

Agent/Customer labels in predictions are reference-assisted.

In [14]:
# ============================================================
# Cell 13 — Reference vs Predicted Speaker Comparison
# ============================================================

def compare_speaker_timelines(call_id, start, end):

    row = selected_calls.loc[
        selected_calls["call_id"] == call_id
    ].iloc[0]

    # Load saved results
    with open(
        OUT_DIR / f"{call_id}.json",
        encoding="utf-8"
    ) as f:
        result = json.load(f)

    # Load reference annotations
    reference = load_reference(row)

    role_map = result["speaker_to_role"]

    records = []

    # Reference speaker segments
    for segment in reference["segments"]:

        seg_start = max(float(segment["start"]), start)
        seg_end = min(float(segment["end"]), end)

        if seg_end > seg_start:

            records.append({
                "start": round(seg_start, 2),
                "end": round(seg_end, 2),
                "source": "Reference",
                "speaker": segment["role"]
            })

    # Predicted speaker segments
    for segment in result["predicted_diarization"]:

        seg_start = max(float(segment["start"]), start)
        seg_end = min(float(segment["end"]), end)

        if seg_end > seg_start:

            records.append({
                "start": round(seg_start, 2),
                "end": round(seg_end, 2),
                "source": "Pyannote",
                "speaker": role_map.get(
                    segment["speaker"],
                    "Unknown"
                )
            })

    comparison = pd.DataFrame(records)

    comparison = comparison.sort_values(
        ["start", "source"]
    ).reset_index(drop=True)

    print("=" * 65)
    print(f"SPEAKER COMPARISON — {call_id}")
    print(f"Time Window: {start}s - {end}s")
    print("=" * 65)

    display(comparison)

    return comparison


# Analyze the worst windows

comparison_005 = compare_speaker_timelines(
    "CALL_005", 300, 330
)

comparison_006 = compare_speaker_timelines(
    "CALL_006", 570, 600
)

SPEAKER COMPARISON — CALL_005
Time Window: 300s - 330s


,start,end,source,speaker
0,300.00,301.14,Reference,agent
1,300.09,302.38,Reference,customer
2,300.17,300.24,Pyannote,agent
3,300.24,301.00,Pyannote,customer
4,303.94,311.54,Reference,agent
5,304.12,305.44,Pyannote,customer
6,307.49,309.60,Pyannote,customer
7,309.00,309.74,Reference,customer
8,310.33,311.49,Pyannote,customer
9,311.94,317.28,Reference,customer


SPEAKER COMPARISON — CALL_006
Time Window: 570s - 600s


,start,end,source,speaker
0,570.00,570.76,Pyannote,customer
1,570.00,570.74,Reference,agent
2,574.54,581.74,Reference,agent
3,574.62,575.03,Pyannote,customer
4,576.02,581.80,Pyannote,customer
5,584.34,588.24,Reference,agent
6,584.38,586.08,Pyannote,customer
7,587.86,588.19,Pyannote,customer


## Cell 14 — Key Findings & Limitations

### 1. Overall Performance

The Pyannote Community-1 model was evaluated on 12 customer service calls covering four English accent groups: American, British, Indian, and Australian.

The model achieved a **macro-average Diarization Error Rate (DER) of 19.93%**, with substantial variation across individual calls.

* **Best performance:** CALL_011 — DER: 6.63%
* **Worst performance:** CALL_005 — DER: 55.03%
* **Second-highest error:** CALL_006 — DER: 50.90%

All 12 calls were successfully processed without execution failures.

### 2. Error Analysis

Detailed error analysis was conducted on CALL_005 and CALL_006.

| Error Component   | CALL_005 | CALL_006 |
| ----------------- | -------: | -------: |
| Missed Detection  |   23.14% |   16.94% |
| False Alarm       |    0.53% |    0.53% |
| Speaker Confusion |   31.35% |   33.43% |
| Total DER         |   55.03% |   50.90% |

**Speaker Confusion was the largest contributor to DER in both calls.**

The model frequently assigned speech to the incorrect speaker, while false alarms contributed relatively little to the overall error.

### 3. Reference vs. Predicted Speakers

A detailed comparison of reference annotations and predicted speaker segments revealed several speaker attribution errors.

**CALL_005:**

Between approximately 304 and 317 seconds, the model repeatedly confused the agent and customer speakers. Additional speaker attribution errors were observed between 320 and 329 seconds.

**CALL_006:**

Between approximately 570 and 588 seconds, several speech segments labeled as Agent in the reference annotations were incorrectly attributed to Customer.

These observations are consistent with the high speaker-confusion error rates.

### 4. Limitations

The following limitations should be considered:

* The evaluation included only 12 calls, limiting the generalizability of the findings.
* Performance varied substantially between calls.
* Speaker overlap and interruptions can complicate speaker attribution.
* Whisper transcription errors may affect the quality of speaker-aligned transcripts.
* Customer and Agent role labels were assigned using reference-assisted mapping rather than independently predicted by Pyannote.
* The evaluation used a zero-second collar and included overlapping speech.
* The analysis identifies where errors occurred but does not establish their underlying acoustic causes.

### 5. Conclusion

Pyannote Community-1 successfully generated speaker diarization predictions for all 12 calls and enabled the creation of speaker-attributed transcripts when combined with Whisper.

Although the model demonstrated promising performance on several calls, the results revealed significant speaker attribution errors in CALL_005 and CALL_006.

The findings highlight the importance of evaluating diarization systems beyond overall performance metrics and examining individual failure cases.

**The current outputs will be retained for subsequent call summarization and sentiment analysis, with their identified limitations documented.**


## Cell 15 — Save Failure Analysis Results

Save reference-versus-predicted speaker comparisons for the two highest-error calls.

In [15]:
# ============================================================
# Cell 15 — Save Failure Analysis Results
# ============================================================

# Save CALL_005 comparison
comparison_005.to_csv(
    OUT_DIR / "CALL_005_speaker_comparison.csv",
    index=False
)

# Save CALL_006 comparison
comparison_006.to_csv(
    OUT_DIR / "CALL_006_speaker_comparison.csv",
    index=False
)

print("=" * 60)
print("FAILURE ANALYSIS RESULTS SAVED")
print("=" * 60)

print("CALL_005 speaker comparison: Saved")
print("CALL_006 speaker comparison: Saved")

print("Failure analysis breakdown: Saved")
print("Window error analysis: Saved")
print("Worst error windows: Saved")

print("=" * 60)

FAILURE ANALYSIS RESULTS SAVED
CALL_005 speaker comparison: Saved
CALL_006 speaker comparison: Saved
Failure analysis breakdown: Saved
Window error analysis: Saved
Worst error windows: Saved
